<a href="https://colab.research.google.com/github/natyherreraa/Pre-Entrega-4-API/blob/main/Pre_Entrega_4_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrega 4 – Extracción de datos de seguros mediante la API de FRED

Este notebook trabaja con la API de FRED para obtener la evolución mensual del índice de precios de los seguros en España durante 2024.

El resultado de la extracción es un archivo CSV con una fila por mes y el valor del índice correspondiente.

In [28]:
# Importación de librerias
import requests
import json
import pandas as pd
import os

%pip -q install python-dotenv



In [29]:
# Configuración de credenciales

from dotenv import load_dotenv

load_dotenv(".env")

api_key = os.getenv("FRED_API_KEY")
print("Clave cargada:", bool(api_key))

Clave cargada: True


In [30]:
print("Archivo .env presente:", os.path.isfile(".env"))

Archivo .env presente: True


In [31]:

# Extracción de datos
url = "https://api.stlouisfed.org/fred/series/observations"

parametros = {
    "api_key": api_key,
    "series_id": "CP1254ESM086NEST",
    "observation_start": "2024-01-01",
    "observation_end": "2024-12-31",
    "file_type": "json",
}



observaciones = []

try:
    respuesta = requests.get(url, params=parametros, timeout=10)
    print("Status code:", respuesta.status_code)

    if respuesta.status_code == 200:
        datos = respuesta.json()
        observaciones = datos["observations"]
        print("Cantidad de observaciones:", len(observaciones))
    elif respuesta.status_code >= 500:
        print("Error en servidor. Volver a intentar más tarde.")
    else:
        print("FRED rechazó la consulta. Revisá la clave y los parámetros.")

except requests.Timeout:
    print("FRED tardó demasiado en responder. Volver a intentar.")
except requests.ConnectionError:
    print("No se pudo conectar con FRED. Revisar conexión a internet.")
except requests.RequestException:
    print("No se pudo completar la consulta a FRED.")

Status code: 200
Cantidad de observaciones: 12


In [32]:
# Exploración de la respuesta de FRED
if observaciones:
    print("Tipo de respuesta:", type(respuesta))
    print("Tipo de datos después de .json():", type(datos))
    print("Primera observación:", observaciones[0])
else:
    print("No se recuperaron observaciones.")

Tipo de respuesta: <class 'requests.models.Response'>
Tipo de datos después de .json(): <class 'dict'>
Primera observación: {'realtime_start': '2026-11-21', 'realtime_end': '2026-11-21', 'date': '2024-01-01', 'value': '118.47'}


In [33]:
# Vista previa de los datos originales

if observaciones:
    print(json.dumps(datos, indent=2))

{
  "realtime_start": "2026-11-21",
  "realtime_end": "2026-11-21",
  "observation_start": "2024-01-01",
  "observation_end": "2024-12-31",
  "units": "lin",
  "output_type": 1,
  "file_type": "json",
  "order_by": "observation_date",
  "sort_order": "asc",
  "count": 12,
  "offset": 0,
  "limit": 100000,
  "observations": [
    {
      "realtime_start": "2026-11-21",
      "realtime_end": "2026-11-21",
      "date": "2024-01-01",
      "value": "118.47"
    },
    {
      "realtime_start": "2026-11-21",
      "realtime_end": "2026-11-21",
      "date": "2024-02-01",
      "value": "118.47"
    },
    {
      "realtime_start": "2026-11-21",
      "realtime_end": "2026-11-21",
      "date": "2024-03-01",
      "value": "118.47"
    },
    {
      "realtime_start": "2026-11-21",
      "realtime_end": "2026-11-21",
      "date": "2024-04-01",
      "value": "118.47"
    },
    {
      "realtime_start": "2026-11-21",
      "realtime_end": "2026-11-21",
      "date": "2024-05-01",
      "va

In [34]:

tabla = pd.DataFrame(observaciones)
display(tabla.head(3))

,realtime_start,realtime_end,date,value
0,2026-11-21,2026-11-21,2024-01-01,118.47
1,2026-11-21,2026-11-21,2024-02-01,118.47
2,2026-11-21,2026-11-21,2024-03-01,118.47


In [35]:
# Trasnformación de fecha e indice a números

resultado = pd.DataFrame()

if len(tabla) > 0:
    resultado["mes"] = tabla["date"].str[:7]
    resultado["indice_precio_seguros"] = pd.to_numeric(tabla["value"])
    display(resultado)
else:
    print("No hay datos para transformar.")

,mes,indice_precio_seguros
0,2024-01,118.47
1,2024-02,118.47
2,2024-03,118.47
3,2024-04,118.47
4,2024-05,118.47
5,2024-06,130.37
6,2024-07,130.37
7,2024-08,130.37
8,2024-09,130.37
9,2024-10,130.37


In [36]:
#Control de calidad de los datos

if len(resultado) > 0:
    print("Cantidad de filas:", len(resultado))
    print("Valores nulos por columna:")
    print(resultado.isna().sum())
    print("Meses duplicados:", resultado["mes"].duplicated().sum())
else:
    print("No hay datos para revisar.")

Cantidad de filas: 12
Valores nulos por columna:
mes                      0
indice_precio_seguros    0
dtype: int64
Meses duplicados: 0


In [38]:
# Exportación del resultado

if len(resultado) > 0:
    resultado.to_csv("data_extracted.csv", index=False)
    print("Archivo guardado: data_extracted.csv")
    print("Filas guardadas:", len(resultado))
else:
    print("No hay datos para guardar.")

Archivo guardado: data_extracted.csv
Filas guardadas: 12
